In [30]:
import re

import numpy as np
import pandas as pd

In [31]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5641 entries, 0 to 5640
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date_start  5641 non-null   object
 1   date_end    602 non-null    object
 2   event       5641 non-null   object
dtypes: object(3)
memory usage: 132.3+ KB


In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

ru_stopwords = stopwords.words("russian")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


texts = df["event"].dropna().astype(str).apply(clean_text)

vectorizer = TfidfVectorizer(
    max_df=0.9,
    min_df=5,
    stop_words=ru_stopwords,
    ngram_range=(1, 3),
    sublinear_tf=True,
)
X = vectorizer.fit_transform(texts)

n_topics = 10
model = NMF(n_components=n_topics, random_state=42)
W = model.fit_transform(X)
H = model.components_

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic in enumerate(H):
    top_words = [feature_names[j] for j in topic.argsort()[:-11:-1]]
    topics[f"Topic {i + 1}"] = top_words

print(len(vectorizer.vocabulary_))
topics

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ruslan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


3313
{'Topic 1': ['парламентские выборы', 'парламентские', 'выборы', 'досрочные парламентские', 'досрочные парламентские выборы', 'досрочные', 'партия', 'большинство', 'сирии', 'мест'], 'Topic 2': ['человек', 'погибли', 'погибли человек', 'результате', 'человека', 'человек погибли', 'получили', 'ранения', 'ранены', 'погибло'], 'Topic 3': ['должность', 'вступил', 'должность президента', 'президента', 'вступил должность', 'вступил должность президента', 'года', 'президент', 'должность президент', 'вступил должность президент'], 'Topic 4': ['тур', 'выборов', 'президентских', 'президентских выборов', 'второй', 'второй тур', 'тур президентских', 'тур президентских выборов', 'второй тур президентских', 'одержал'], 'Topic 5': ['мира', 'чемпионат', 'чемпионат мира', 'россия', 'мира хоккею', 'хоккею', 'чемпионат мира хоккею', 'сборная', 'шайбой', 'хоккею шайбой'], 'Topic 6': ['премьер', 'министром', 'премьер министром', 'стал', 'новым', 'новым премьер', 'новым премьер министром', 'министра', 'п

In [34]:
from sklearn.decomposition import TruncatedSVD

n_topics = 10

lsa = TruncatedSVD(
    n_components=n_topics,
    random_state=42
)

X_lsa = lsa.fit_transform(X)
feature_names = vectorizer.get_feature_names_out()

topics = {}

for i, comp in enumerate(lsa.components_):
    indices = np.argsort(np.abs(comp))[-12:]
    top_words = [feature_names[j] for j in indices]
    topics[f"Topic {i + 1}"] = top_words

topics

{'Topic 1': ['тур',
  'победу одержала',
  'победу одержал',
  'одержал',
  'одержала',
  'партия',
  'президентские выборы',
  'президентские',
  'победу',
  'парламентские выборы',
  'парламентские',
  'выборы'],
 'Topic 2': ['президента',
  'погибло',
  'получили ранения',
  'ранены',
  'ранения',
  'получили',
  'человек погибли',
  'человека',
  'результате',
  'погибли человек',
  'погибли',
  'человек'],
 'Topic 3': ['одержал',
  'погибли',
  'человек',
  'тур',
  'выборов',
  'парламентские',
  'парламентские выборы',
  'вступил должность',
  'должность президента',
  'вступил',
  'президента',
  'должность'],
 'Topic 4': ['одержал',
  'победу',
  'второй',
  'президентских выборов',
  'второй тур',
  'президентских',
  'вступил должность',
  'должность президента',
  'выборов',
  'тур',
  'вступил',
  'должность'],
 'Topic 5': ['шайбой',
  'мира хоккею шайбой',
  'хоккею шайбой',
  'одержала',
  'чемпионат мира хоккею',
  'мира хоккею',
  'хоккею',
  'сборная',
  'россия',
  '

In [35]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=30,
    doc_topic_prior=0.1,  # alpha
    topic_word_prior=0.01  # beta
)
X_lda = lda.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic_dist in enumerate(lda.components_):
    top_idx = topic_dist.argsort()[-12:][::-1]
    topics[f"Topic {i + 1}"] = [feature_names[j] for j in top_idx]

topics

{'Topic 1': ['сша',
  'союз',
  'корабля',
  'экипаж',
  'тма',
  'союз тма',
  'космического',
  'космического корабля',
  'корабля союз',
  'владимир',
  'космический',
  'путин'],
 'Topic 2': ['премьер',
  'министром',
  'премьер министром',
  'стал',
  'отставку',
  'министр',
  'премьер министр',
  'новым',
  'президент',
  'новым премьер',
  'новым премьер министром',
  'лидер'],
 'Topic 3': ['космодрома',
  'запуск',
  'открытие',
  'мире',
  'казахстан',
  'официально',
  'саммит',
  'впервые',
  'стала',
  'байконур',
  'космодрома байконур',
  'истории'],
 'Topic 4': ['россии',
  'начало',
  'территории',
  'кндр',
  'первого',
  'сша',
  'лет',
  'стран',
  'сирии',
  'решение',
  'государств',
  'власти'],
 'Topic 5': ['выборы',
  'парламентские',
  'парламентские выборы',
  'победу',
  'президентские',
  'президентские выборы',
  'партия',
  'одержал',
  'победу одержал',
  'президента',
  'тур',
  'выборов'],
 'Topic 6': ['убийство',
  'открыт',
  'нато',
  'должность пре

In [36]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

texts = df["event"]

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    min_df=5
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts)

2025-12-18 21:31:05,523 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2025-12-18 21:31:45,340 - BERTopic - Embedding - Completed ✓
2025-12-18 21:31:45,340 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-18 21:31:46,642 - BERTopic - Dimensionality - Completed ✓
2025-12-18 21:31:46,642 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-18 21:31:47,989 - BERTopic - Cluster - Completed ✓
2025-12-18 21:31:47,992 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-18 21:31:48,183 - BERTopic - Representation - Completed ✓


In [37]:
import random
from collections import defaultdict


def sample_docs_per_topic(texts, topics, n_samples=10, seed=42):
    random.seed(seed)

    topic_to_docs = defaultdict(list)
    for text, topic in zip(texts, topics):
        topic_to_docs[topic].append(text)

    for topic_id, docs in sorted(topic_to_docs.items()):
        if topic_id == -1:
            print(f"topic {topic_id} | total docs: {len(docs)}")
            continue

        print("=" * 80)
        print(f"TOPIC {topic_id} | total docs: {len(docs)}")
        print("=" * 80)

        sampled = random.sample(docs, min(n_samples, len(docs)))
        for i, doc in enumerate(sampled, 1):
            print(f"{i}. {doc}")
        print()


def save_topics_barchart(topic_model: BERTopic, out_html="topics_barchart.html", top_n_topics=30):
    """
    Сохраняет интерактивный bar chart с размерами/словами тем (BERTopic).
    """
    fig = topic_model.visualize_barchart(top_n_topics=top_n_topics, n_words=10)
    fig.write_html(out_html)
    return fig


def save_documents_scatter(topic_model, texts, topics, out_html="documents_scatter.html"):
    """
    Сохраняет интерактивный scatter документов по темам (BERTopic).
    """
    fig = topic_model.visualize_documents(docs=texts, topics=topics)
    fig.write_html(out_html)
    return fig

In [38]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1492,-1_на_выборов_тур_победу,"[на, выборов, тур, победу, выборы, парламентск...",[первый тур президентских выборов в Бенине. Во...
1,0,186,0_перу_всеобщие_чили_президента,"[перу, всеобщие, чили, президента, всеобщие вы...",[Второй тур президентских выборов в Бразилии. ...
2,1,183,1_самолёт_борту_на борту_все,"[самолёт, борту, на борту, все, потерпел, ката...",[самолёт Airbus A321 российской авиакомпании «...
3,2,116,2_чемпионат_чемпионат мира_чемпионат мира по_м...,"[чемпионат, чемпионат мира, чемпионат мира по,...","[чемпионат мира по тяжёлой атлетике (Хьюстон, ..."
4,3,113,3_ес_союза_председателем_стала,"[ес, союза, председателем, стала, совета, евро...",[Испания стала государством-председателем Сове...
5,4,108,4_казахстана_казахстане_кыргызстана_армении,"[казахстана, казахстане, кыргызстана, армении,...",[Президентские выборы в Казахстане. Победил де...
6,5,90,5_результате взрыва_взрыва_результате_человек,"[результате взрыва, взрыва, результате, челове...",[в результате взрыва пиротехники на рынке в Би...
7,6,88,6_сша_президент сша_джордж_джо,"[сша, президент сша, джордж, джо, представител...","[промежуточные выборы сенаторов США., Марш мил..."
8,7,80,7_произошло_землетрясения_человек_более,"[произошло, землетрясения, человек, более, без...",[В Тбилиси произошло землетрясение магнитудой ...
9,8,78,8_россии на украину_украину_на украину_россии на,"[россии на украину, украину, на украину, росси...",[Вторжение России на Украину: окончание боёв з...


In [39]:
sample_docs_per_topic(texts, topics, n_samples=10)
save_topics_barchart(topic_model, out_html="topic_plots/topics_barchart_auto.html")
save_documents_scatter(topic_model, texts, topics, out_html="topic_plots/documents_scatter_auto.html")

topic -1 | total docs: 1492
TOPIC 0 | total docs: 186
1. Всеобщие выборы в Сальвадоре. Действующий президент Найиб Букеле победил на выборах, набрав более 80 % голосов, став первым президентом, переизбранным в Сальвадоре с 1944 года.
2. в Венесуэле совершена попытка государственного переворота. Свергнут президент Уго Чавес, распущены парламент и Верховный суд. Временным президентом стал Педро Кармона. На следующий день Чавес восстановлен в должности, а Кармона арестован.
3. Альберто Фухимори переизбран президентом Перу. Оппозиция не признала выборы и ответила новыми маршами протеста, в которых участвовали сотни тысяч людей. Полиция жестоко подавляла выступления. В ходе столкновений сотни людей были ранены и арестованы.
4. парламентские выборы на Кубе.
5. В Перу прошёл второй тур президентских выборов. Победил левый кандидат Ольянта Умала.
6. в Буэнос-Айресе стартовало ралли «Дакар-2010».
7. Международный валютный фонд одобрил выделение Бразилии займа в размере $30,4 миллиарда долларов 

In [40]:
# semi-supervised learning
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
    zeroshot_topic_list=[
        'природная катастрофа', 'авиакатастрофа', 'государственный переворот', 'вооруженный конфликт', 'теракт', 'протесты', 'санкции', 'спорт', 'закон'
    ],
    seed_topic_list=[
        ["землетрясение", "цунами", "извержение вулкана", "ураган", "тайфун",
         "наводнение", "оползень", "сель", "засуха", "лесной пожар"],
        ["авиакатастрофа", "крушение самолета", "пассажирский самолет",
         "на борту", "рейс", "экипаж"],
        ["государственный переворот", "госпереворот", "военный переворот",
 "свержение власти", "захват власти", "путч"],
        ["вооруженный конфликт", "война", "военные действия", "наступление", "обстрел", "ВС РФ", "ВСУ"],
        ["теракт", "смертник", "террористический акт"],
        ["санкции", "пакет санкций"],
        ["акция протеста", "протест", "массовые протесты", "беспорядки"],
        ["запуск ракеты", "космос", "спутник", "космический аппарат", "орбита", "космодром"],
        ["чемпионат мира", "спорт", "золотая медаль", "олимпийские игры", "сборная"],
        ["выборы президента", "выборы премьер-министра", "парламентские выборы"],
        ["закон", "подписание закона", "вступление в силу закона", "законопроект", "принятие закона"],
        ["Nvidia", "Microsoft", "Google", "Samsung", "Huawei", "Facebook", "Apple"]
    ]
)

topics, probs = topic_model.fit_transform(texts)

2025-12-18 21:32:29,300 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2025-12-18 21:33:07,943 - BERTopic - Embedding - Completed ✓
2025-12-18 21:33:07,943 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-18 21:33:08,049 - BERTopic - Guided - Completed ✓
2025-12-18 21:33:08,050 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-18 21:33:09,610 - BERTopic - Dimensionality - Completed ✓
2025-12-18 21:33:09,611 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2025-12-18 21:33:09,649 - BERTopic - Zeroshot Step 1 - Completed ✓
2025-12-18 21:33:10,973 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-18 21:33:11,851 - BERTopic - Cluster - Completed ✓
2025-12-18 21:33:11,852 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2025-12-18 21:33:11,863 - BERTopic - Zeroshot Step 2 - Completed ✓
2025-12-18 21:33:11,863 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-18 21:33:12,051 - BERTopic - Representation - Completed ✓


In [41]:
topic_model.reduce_topics(
    texts,
    nr_topics=15
)

topics, probs = topic_model.transform(texts)
topic_model.get_topic_info()

2025-12-18 21:33:12,282 - BERTopic - Topic reduction - Reducing number of topics
2025-12-18 21:33:12,346 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-18 21:33:12,503 - BERTopic - Representation - Completed ✓
2025-12-18 21:33:12,504 - BERTopic - Topic reduction - Reduced number of topics from 97 to 15


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2025-12-18 21:33:50,245 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1361,-1_по_на_выборы_мира,"[по, на, выборы, мира, президента, президент, ...",[Второй тур президентских выборов в Литве. Поб...
1,0,1023,0_выборы_победу_премьер_выборов,"[выборы, победу, премьер, выборов, президента,...",[выборы президента Замбии. Победу одержал канд...
2,1,1008,1_человек_погибли_результате_на,"[человек, погибли, результате, на, более, поги...",[в результате стрельбы в Копенгагене (Дания) п...
3,2,703,2_россии_война_вторая_на,"[россии, война, вторая, на, россия, рф, россий...",[Вторая чеченская война: вертолёт Ми-8МТ МВД Р...
4,3,338,3_мира_по_саммит_международный,"[мира, по, саммит, международный, состоялся, п...","[чемпионат мира по ралли (Австралия)., Кубок м..."
5,4,307,4_погибли_человек_все_человека,"[погибли, человек, все, человека, на, потерпел...",[В результате возгорания поезда-фуникулёра на ...
6,5,306,5_союз_посадки_сша_станции,"[союз, посадки, сша, станции, запуск, ракеты, ...",[приземление корабля «Союз ТМА-1». Экипаж поса...
7,6,149,6_военный_президента_президент_конго,"[военный, президента, президент, конго, чен, ю...","[В Судане произошёл военный переворот, в ходе ..."
8,7,136,7_закон_принятие_против_силу,"[закон, принятие, против, силу, сша, президент...",[Принятие пятого пакета санкций против России....
9,8,114,8_компания_выход_системы_сша,"[компания, выход, системы, сша, китая, мире, п...",[Компания «Microsoft» выпустила операционную с...


In [42]:
sample_docs_per_topic(texts, topics, n_samples=10)

topic -1 | total docs: 199
TOPIC 0 | total docs: 1214
1. Светозара Маровича избрали президентом Государственного Союза Сербии и Черногории. Согласно конституционной хартии он одновременно стал премьер-министром.
2. Аскар Акаев переизбран президентом Республики Кыргызстан на третий срок;
3. новым премьер-министром Чехии стал Богуслав Соботка.
4. парламентские и президентские выборы в Парагвае. Президентом Парагвая избран Орасио Картес.
5. второй тур выборов президента Египта. Кандидат от исламского движения «Братья-мусульмане» Мухаммед Мурси победил, набрав 52 % голосов.
6. Джордж Буш (младший) вступил в должность президента США на второй срок.
7. В Южной Кореи состоялись президентские выборы. Победу одержал Но Му Хён (48,9 % голосов избирателей; вступил в должность президента 25 февраля 2003 года). Второе место занял Ли Хве Чхан (46,58 % голосов).
8. Парламентские выборы в Южной Осетии.
9. на конституционном референдуме в Азербайджане принят ряд поправок в основной закон, связанных в т

In [43]:
save_topics_barchart(topic_model, out_html="topic_plots/topics_barchart_15_topics.html")
save_documents_scatter(topic_model, texts, topics, out_html="topic_plots/documents_scatter_15_topics.html")